In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import QuantileTransformer
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [3]:

# ----------- Basisverzeichnis (aktuelles Notebook-Verzeichnis) ---------------
base_dir = Path.cwd()

# ----------- Pfad zur "Kompletten Datenbank" setzen --------------
# AKTUALISIERT: Liest aus 'Nach_IBF-und-Temp_Filter' (Qualitätsdaten)
db_root = (
    base_dir.parent.parent 
    / "1_Acquisition" 
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken"
    / "Nach_IBF-und-Temp_Filter"
    / "Komplette_Datenbank"
)

# ----------- Alle Unterordner lesen, Ordner mit "neustem" Datum setzen -----------------
if not db_root.exists():
     raise FileNotFoundError(f"Datenbank-Pfad nicht gefunden: {db_root}")

timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Datenbank-Ordner in {db_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

# ----------- Komplette Datenbank als CSV ------------------
input_path = latest_folder / "Komplette_Datenbank.csv"

print("Verwendeter Datenbankpfad:")
print(input_path)


Verwendeter Datenbankpfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_IBF-und-Temp_Filter\Komplette_Datenbank\2026-01-05_23-05-35\Komplette_Datenbank.csv


In [4]:
# ------------- Ordner "Preprocessing" zum Abspeichern --------------
output_root = base_dir / "Preprocessing"
output_root.mkdir(exist_ok=True) 

# ------------- Zeitstempel erzeugen ----------------
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# ------------- Neuen Ordner mit Zeitstempel erzeugen --------------
output_dir = output_root / timestamp
output_dir.mkdir(parents=True, exist_ok=False)

print("Erzeugter Output-Ordner:")
print(output_dir)

Erzeugter Output-Ordner:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-01-05_23-05-46


<div class="alert alert-info">
    <b>Preprocessing & Scaling</b></b><br><br>
    <b>1. Log-Transformation</b><br>
    - Anwendung von np.log1p(x) (ln(1 + x)).<br>
    - Verhindert negative Werte für kleine Zahlen (x < 1)<br>
    <br>
    <b>2. Robust Scaling (IQR)</b><br>
    - Zentriert auf Median, skaliert mit IQR (Ausreißer-resistent).<br>
    - Macht Daten statistisch ideal für die SOM<br>
</div>

In [5]:
# ------------ Daten laden ---------------
df = pd.read_csv(input_path, low_memory=False)

print(f"Datensatz geladen mit {df.shape[0]} Zeilen und {df.shape[1]} Spalten.")

Datensatz geladen mit 94264 Zeilen und 31 Spalten.


In [6]:
# ----------------- Spalten ausschließen -----------------
EXCLUDE_COLS = [
    "WGS84_latitude",
    "WGS84_longitude",
    "Database_number",
    "id",                 
    "index",
    "temperature_in_c",
    "rock_type",
]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
transform_candidates = [c for c in numeric_cols if c not in EXCLUDE_COLS]

print(f"{len(transform_candidates)} numerische Spalten sollten transformiert werden")

19 numerische Spalten sollten transformiert werden


In [7]:
# ----------------- Log-Transformation Report -----------------

skewness = df[transform_candidates].skew().sort_values(ascending=False)
skewed_cols = skewness[abs(skewness) > 1.0].index.tolist()

df_base = df.copy()
base_cols = []
applied_log_cols = []

pdf_log_path = output_dir / "Log_Transformation_Report.pdf"

with PdfPages(pdf_log_path) as pdf:
    fig = plt.figure(figsize=(10, 6))
    
    # ------------ Detaillierte Beschreibung ---------------
    plt.text(0.5, 0.5, f"Log-Transformation Report\n\nMethode: Natural Logarithm (np.log1p)\nFormel: f(x) = ln(1 + x)\n\nZiel: Vorbehandlung extremer Werte ohne negative Ergebnisse", 
             ha='center', va='center', fontsize=16)
    plt.axis('off')
    pdf.savefig(fig)
    plt.close()

    for col in transform_candidates:
        if df[col].min() >= 0 and col in skewed_cols:
            new_col = f"{col}_log"
            
            # --- log1p, damit Werte nicht negativ werden ---
            df_base[new_col] = np.log1p(df[col])
            base_cols.append(new_col)
            applied_log_cols.append(new_col)
            
            # --- Visualisierung ---
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            sns.histplot(df[col].dropna(), bins=30, ax=axes[0], color="skyblue", kde=True)
            axes[0].set_title(f"Orig: {col}")
            sns.histplot(df_base[new_col].dropna(), bins=30, ax=axes[1], color="lightgreen", kde=True)
            
            # --- Angabe der Transfomration im Titel ---
            axes[1].set_title(f"Transformed: ln(1 + {col})")
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
        else:
            base_cols.append(col)

print(f"Log-Report erstellt: {pdf_log_path}")

Log-Report erstellt: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-01-05_23-05-46\Log_Transformation_Report.pdf


In [8]:
# ----------------- Gaussian Scaling (QuantileTransformer) -----------------

print('STARTING GAUSSIAN SCALING EXECUTION')
results = []
df_final = df_base.copy()
final_features = []

pdf_report_path = output_dir / "Gaussian_Scaling_Report.pdf"

with PdfPages(pdf_report_path) as pdf:
    
    fig = plt.figure(figsize=(10, 6))
    
    # ----------------- Beschreibung -----------------
    plt.text(0.5, 0.5, f"Gaussian Scaling Report (Quantile Transformation)\n\nMethode: QuantileTransformer (output_distribution='normal')\nZiel: Erzwingt Normalverteilung (Gaussian).\nIdeal fur SOMs auf schiefen geologischen Daten.", 
             ha='center', va='center', fontsize=16)
    plt.axis('off')
    pdf.savefig(fig)
    plt.close()

    for col in base_cols:
        series = df_base[col].dropna()
        if len(series) == 0: continue
        
        vals = series.values.reshape(-1, 1)
        
        # ----------- Transform -------------
        # Nutze QuantileTransformer fur Gaussianization
        scaler = QuantileTransformer(output_distribution='normal', random_state=42)
        trans_vals = scaler.fit_transform(vals).flatten()

        # ----------- Plot -------------
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(series, bins=30, ax=axes[0], color="lightgreen", kde=True)
        axes[0].set_title(f"Input: {col}")
        sns.histplot(trans_vals, bins=30, ax=axes[1], color="crimson", kde=True)
        
        # ---------- Explizite Angabe ------------
        axes[1].set_title(f"Output: Gaussian Scaled\n(Normal Distribution)")
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        
        # ---------- Speichern ---------------
        full_series = df_base[col]
        valid_mask = full_series.notna()
        
        if valid_mask.sum() > 0:
            scaler_col = QuantileTransformer(output_distribution='normal', random_state=42)
            final_trans = scaler_col.fit_transform(full_series.dropna().values.reshape(-1, 1)).flatten()
            
            # Suffix '_gauss'
            new_col_name = f"{col}_gauss"
            df_final.loc[valid_mask, new_col_name] = final_trans
            final_features.append(new_col_name)

print(f"Fertig. Report: {pdf_report_path}")

STARTING GAUSSIAN SCALING EXECUTION


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (397). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (397). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (257). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (257). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (373). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (373). n_quantiles is set to n_samples.
  warnings.warn(
C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (12). n_quantiles is set to n_samples.
  warnings.warn(


Fertig. Report: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-01-05_23-05-46\Gaussian_Scaling_Report.pdf


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (12). n_quantiles is set to n_samples.
  warnings.warn(


In [9]:
# ----------------- Als .csv speichern -----------------
output_file_som = output_dir / "Preprocessed_SOM_Ready.csv"


# ----------------- Nur finalisierte Spalten speichern -------------
final_cols_to_save = EXCLUDE_COLS + final_features
final_cols_real = [c for c in final_cols_to_save if c in df_final.columns]

df_final[final_cols_real].to_csv(output_file_som, index=False)
print(f"SOM Data gespeichert: {output_file_som}")

SOM Data gespeichert: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-01-05_23-05-46\Preprocessed_SOM_Ready.csv
